In [15]:
import cv2
import os
import shutil
import numpy as np
from ultralytics import YOLO

def process_subject_folder_to_silhouettes(
    input_folder="new raw dataset/Ankit", 
    output_folder="live_test_folder", 
    img_size=(64, 64),
    
    # === TWEAK THESE PARAMETERS ===
    inference_size=1080,        # Increase to 1280 or 1920 for maximum sharpness (runs slower)
    confidence_threshold=0.8,   # Increase to 0.6 or 0.7 for stricter, cleaner edges
    extract_every_n_frame=3     # 1 = 30 FPS, 2 = 15 FPS, 3 = 10 FPS, 6 = 5 FPS
    # ==============================
):
    # 1. Clean and prepare the output folder once
    if os.path.exists(output_folder):
        shutil.rmtree(output_folder)
    os.makedirs(output_folder)
    
    # 2. Load the YOLO model once for all videos
    print("Loading YOLOv8 Segmentation Model...")
    model = YOLO("yolov8n-seg.pt")
    
    total_saved_count = 0
    
    # 3. Iterate through every file in the subject's folder
    valid_extensions = ('.mp4', '.avi', '.mov', '.mkv')
    video_files = [f for f in os.listdir(input_folder) if f.lower().endswith(valid_extensions)]
    
    if not video_files:
        print(f"No video files found in {input_folder}")
        return

    print(f"Found {len(video_files)} videos in '{input_folder}'. Starting processing...\n")

    for video_name in video_files:
        video_path = os.path.join(input_folder, video_name)
        video_base_name = os.path.splitext(video_name)[0] # Get name without extension
        
        cap = cv2.VideoCapture(video_path)
        if not cap.isOpened():
            print(f"Error: Could not open {video_path}. Skipping.")
            continue
            
        frame_count = 0
        video_saved_count = 0
        print(f"Processing '{video_name}'...")

        while cap.isOpened():
            ret, frame = cap.read()
            if not ret: 
                break
                
            frame_count += 1
            
            # Frame Skipping Logic
            if frame_count % extract_every_n_frame != 0:
                continue
            
            # YOLO Inference
            results = model.predict(
                frame, 
                classes=[0], 
                retina_masks=True, 
                imgsz=inference_size, 
                conf=confidence_threshold,
                verbose=False
            )
            
            binary_silhouette = np.zeros((frame.shape[0], frame.shape[1]), dtype=np.uint8)
            
            for r in results:
                if r.masks is not None:
                    mask_data = r.masks.data[0].cpu().numpy()
                    binary_silhouette = np.maximum(binary_silhouette, (mask_data > 0.5).astype(np.uint8) * 255)

            if np.sum(binary_silhouette) > 0:
                x, y, w, h = cv2.boundingRect(binary_silhouette)
                
                pad_w = int(w * 0.10)
                pad_h = int(h * 0.10)
                
                x1 = max(x - pad_w, 0)
                y1 = max(y - pad_h, 0)
                x2 = min(x + w + pad_w, frame.shape[1])
                y2 = min(y + h + pad_h, frame.shape[0])
                
                cropped_person = binary_silhouette[y1:y2, x1:x2]
                final_silhouette = cv2.resize(cropped_person, img_size)
                
                # Save with a smart name: videoName_frame_0001.png
                save_filename = f"{video_base_name}_frame_{video_saved_count:04d}.png"
                save_path = os.path.join(output_folder, save_filename)
                
                cv2.imwrite(save_path, final_silhouette)
                video_saved_count += 1
                total_saved_count += 1

        cap.release()
        print(f"  -> Extracted {video_saved_count} silhouettes from '{video_name}'.")

    print(f"\n[SUCCESS] Finished folder! Extracted a total of {total_saved_count} silhouettes to '{output_folder}'.")

if __name__ == "__main__":
    current_dir = os.getcwd()
    
    # Point this to the folder containing all videos of the subject
    absolute_input_folder = os.path.join(current_dir, r"D:\Study\Coding\Project\GAIT_2\New Dataset\raw dataset", "Tiwari") 
    absolute_output_folder = os.path.join(current_dir, r"D:\Study\Coding\Project\GAIT_2\New Dataset\processed\Tiwari")
    
    process_subject_folder_to_silhouettes(
        input_folder=absolute_input_folder, 
        output_folder=absolute_output_folder, 
        img_size=(64, 64),
        inference_size=1080,       
        extract_every_n_frame=2    
    )

Loading YOLOv8 Segmentation Model...
Found 10 videos in 'D:\Study\Coding\Project\GAIT_2\New Dataset\raw dataset\Tiwari'. Starting processing...

Processing '20260322_145418.mp4'...
WARNING imgsz=[1080] must be multiple of max stride 32, updating to [1088]
WARNING imgsz=[1080] must be multiple of max stride 32, updating to [1088]
WARNING imgsz=[1080] must be multiple of max stride 32, updating to [1088]
WARNING imgsz=[1080] must be multiple of max stride 32, updating to [1088]
WARNING imgsz=[1080] must be multiple of max stride 32, updating to [1088]
WARNING imgsz=[1080] must be multiple of max stride 32, updating to [1088]
WARNING imgsz=[1080] must be multiple of max stride 32, updating to [1088]
WARNING imgsz=[1080] must be multiple of max stride 32, updating to [1088]
WARNING imgsz=[1080] must be multiple of max stride 32, updating to [1088]
WARNING imgsz=[1080] must be multiple of max stride 32, updating to [1088]
WARNING imgsz=[1080] must be multiple of max stride 32, updating to [

In [17]:
import os
import shutil
import random
from collections import defaultdict

def reorganize_dataset(source_root, output_root, validation_split=0.3, random_seed=42):
    """
    CUSTOM DATASET VERSION
    
    Reorganize custom dataset by:
    - Reading all subject folders (regardless of name/number)
    - Splitting images into train/validation (30% validation, 70% training)
    - Organizing by person folder
    - No overlapping between splits
    """

    random.seed(random_seed)

    # Create output directories
    train_root = os.path.join(output_root, 'train')
    validation_root = os.path.join(output_root, 'validation')

    os.makedirs(train_root, exist_ok=True)
    os.makedirs(validation_root, exist_ok=True)

    print(f"Source root: {source_root}")
    print(f"Output root: {output_root}")
    print(f"Validation split: {validation_split * 100}%")
    print("-" * 60)
    print()

    # Get all person folders (Removed the .isdigit() constraint)
    all_items = os.listdir(source_root)
    person_folders = sorted([
        item for item in all_items 
        if os.path.isdir(os.path.join(source_root, item))
    ])

    print(f"Found {len(person_folders)} subject folders: {', '.join(person_folders)}")
    print("-" * 60)
    print()

    # Dictionary to store all images per person
    person_images = defaultdict(list)

    # Walk through each person folder
    for person_name in person_folders:
        person_path = os.path.join(source_root, person_name)

        # Collect all images from this person 
        for root, dirs, files in os.walk(person_path):
            for file in files:
                if file.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp', '.tiff')):
                    full_path = os.path.join(root, file)
                    person_images[person_name].append(full_path)

    # Print summary
    total_images = sum(len(images) for images in person_images.values())
    if total_images == 0:
        print("ERROR: No images found! Please check your source_root path.")
        return None, None
        
    print(f"Total images collected: {total_images:,}")
    print(f"Subjects with images: {len(person_images)}")
    print()
    print("Processing and splitting images:")
    print("-" * 60)

    # Process each person's images
    stats = {'train': 0, 'validation': 0, 'persons_processed': 0}

    for person_name in sorted(person_images.keys()):
        images = person_images[person_name]
        num_images = len(images)

        if num_images == 0:
            continue

        # Randomly shuffle images for this person
        random.shuffle(images)

        # Calculate split point
        split_point = int(num_images * validation_split)

        # Note: validation takes the first portion, train takes the rest
        validation_images = images[:split_point]
        train_images = images[split_point:]

        # Create person folders in train and validation directories
        train_person_dir = os.path.join(train_root, person_name)
        validation_person_dir = os.path.join(validation_root, person_name)

        os.makedirs(train_person_dir, exist_ok=True)
        os.makedirs(validation_person_dir, exist_ok=True)

        # Copy images to train folder
        for img_path in train_images:
            filename = os.path.basename(img_path)
            dest_path = os.path.join(train_person_dir, filename)
            shutil.copy2(img_path, dest_path)
            stats['train'] += 1

        # Copy images to validation folder
        for img_path in validation_images:
            filename = os.path.basename(img_path)
            dest_path = os.path.join(validation_person_dir, filename)
            shutil.copy2(img_path, dest_path)
            stats['validation'] += 1

        stats['persons_processed'] += 1

        # Print progress for each person
        val_count = len(validation_images)
        train_count = len(train_images)
        val_pct = (val_count / num_images) * 100 if num_images > 0 else 0

        print(f"Subject '{person_name}': Total={num_images:6,d} | Train={train_count:6,d} | Val={val_count:6,d} ({val_pct:.1f}%)")

    # Print summary statistics
    print()
    print("=" * 60)
    print("REORGANIZATION COMPLETE")
    print("=" * 60)
    print(f"Subjects processed: {stats['persons_processed']}")
    print(f"Total images in training: {stats['train']:,}")
    print(f"Total images in validation: {stats['validation']:,}")
    print(f"Total images processed: {stats['train'] + stats['validation']:,}")

    if stats['train'] + stats['validation'] > 0:
        actual_val_pct = (stats['validation'] / (stats['train'] + stats['validation'])) * 100
        print(f"Actual validation split: {actual_val_pct:.2f}%")

    print()
    print(f"Train directory: {train_root}")
    print(f"Validation directory: {validation_root}")
    print()

    return train_root, validation_root


if __name__ == "__main__":
    # Ensure these paths match your system
    SOURCE_DATASET = r"D:\Study\Coding\Project\GAIT_2\New Dataset\processed"
    OUTPUT_DIRECTORY = r"D:\Study\Coding\Project\GAIT_2\New Dataset"

    train_dir, val_dir = reorganize_dataset(
        source_root=SOURCE_DATASET,
        output_root=OUTPUT_DIRECTORY,
        validation_split=0.3, # 30% for validation, 70% for training
        random_seed=42
    )

Source root: D:\Study\Coding\Project\GAIT_2\New Dataset\processed
Output root: D:\Study\Coding\Project\GAIT_2\New Dataset
Validation split: 30.0%
------------------------------------------------------------

Found 8 subject folders: Adarsh, Aman, Ankit, Himanshu, Jyotibrat, Kunal, Swayam, Tiwari
------------------------------------------------------------

Total images collected: 6,177
Subjects with images: 8

Processing and splitting images:
------------------------------------------------------------
Subject 'Adarsh': Total=   716 | Train=   502 | Val=   214 (29.9%)
Subject 'Aman': Total=   870 | Train=   609 | Val=   261 (30.0%)
Subject 'Ankit': Total=   796 | Train=   558 | Val=   238 (29.9%)
Subject 'Himanshu': Total=   741 | Train=   519 | Val=   222 (30.0%)
Subject 'Jyotibrat': Total=   866 | Train=   607 | Val=   259 (29.9%)
Subject 'Kunal': Total=   712 | Train=   499 | Val=   213 (29.9%)
Subject 'Swayam': Total=   725 | Train=   508 | Val=   217 (29.9%)
Subject 'Tiwari': Tota

In [21]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import timm
import os
import time
import json
from tqdm import tqdm
from torch.cuda.amp import GradScaler, autocast

# === UPDATE THIS TO YOUR NEW DATASET FOLDER ===
# This should be the folder that contains your 'train' and 'validation' folders
DATA_DIR = r"D:\Study\Coding\Project\GAIT_2\New Dataset" 

BATCH_SIZE = 64
NUM_EPOCHS = 20
LEARNING_RATE = 0.001

def train_gait_model():
    print(f"Initializing custom training process...")

    # --- Set up Data Transformations ---
    data_transforms = {
        'train': transforms.Compose([
            transforms.Resize((72, 72)),
            transforms.ToTensor(),
            transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5])
        ]),
        'validation': transforms.Compose([
            transforms.Resize((72, 72)),
            transforms.ToTensor(),
            transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5])
        ]),
    }

    # --- Load the Datasets ---
    train_dir = os.path.join(DATA_DIR, 'train')
    validation_dir = os.path.join(DATA_DIR, 'validation')

    print(f"Loading data from: {DATA_DIR}")
    
    train_dataset = datasets.ImageFolder(train_dir, data_transforms['train'])
    val_dataset = datasets.ImageFolder(validation_dir, data_transforms['validation'])

    # Dynamically figure out how many subjects you have based on folders
    class_names = train_dataset.classes
    num_classes = len(class_names)
    print(f"\nFound {num_classes} subjects: {class_names}")

    # --- CRITICAL: Save the class mapping for real-time testing later ---
    mapping_path = 'class_mapping.json'
    with open(mapping_path, 'w') as f:
        json.dump(train_dataset.class_to_idx, f)
    print(f"Saved class mapping to {mapping_path}")

    # --- Create DataLoaders ---
    # Note: I reduced num_workers to 4. 8 often causes BrokenPipeErrors on Windows laptops.
    dataloaders = {
        'train': DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=4, pin_memory=True),
        'validation': DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True)
    }

    dataset_sizes = {'train': len(train_dataset), 'validation': len(val_dataset)}
    print(f"Training images: {dataset_sizes['train']}")
    print(f"Validation images: {dataset_sizes['validation']}\n")

    # --- Build the Xception Model using TIMM ---
    # This automatically loads ImageNet weights and changes the final layer to match your num_classes
    print("Loading Xception model via TIMM...")
    model = timm.create_model('xception', pretrained=True, num_classes=num_classes)

    # --- Set up for Training ---
    device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
    print(f"Training will use device: {device}\n")
    model = model.to(device)

    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

    scaler = GradScaler()
    use_amp = torch.cuda.is_available()

    # --- Start the Training Loop ---
    start_time = time.time()
    
    for epoch in range(NUM_EPOCHS):
        print(f'Epoch {epoch+1}/{NUM_EPOCHS}')
        print('-' * 10)

        for phase in ['train', 'validation']:
            if phase == 'train':
                model.train()
            else:
                model.eval()

            running_loss = 0.0
            running_corrects = 0

            progress_bar = tqdm(dataloaders[phase], desc=f'{phase.capitalize()} Phase')
            
            for inputs, labels in progress_bar:
                inputs = inputs.to(device)
                labels = labels.to(device)
                optimizer.zero_grad(set_to_none=True)

                with autocast(enabled=use_amp):
                    outputs = model(inputs)
                    loss = criterion(outputs, labels)
                
                _, preds = torch.max(outputs, 1)

                if phase == 'train':
                    scaler.scale(loss).backward()
                    scaler.step(optimizer)
                    scaler.update()

                running_loss += loss.item() * inputs.size(0)
                running_corrects += torch.sum(preds == labels.data)
            
            epoch_loss = running_loss / dataset_sizes[phase]
            epoch_acc = running_corrects.double() / dataset_sizes[phase]
            print(f'{phase.capitalize()} Loss: {epoch_loss:.4f} Acc: {epoch_acc:.4f}\n')

    time_elapsed = time.time() - start_time
    print(f"Training complete in {time_elapsed // 60:.0f}m {time_elapsed % 60:.0f}s")
    
    # --- Save the Custom Model ---
    model_save_path = f'new_dataset_gait_xception_FULL.pth'
    torch.save(model, model_save_path)
    print(f"Model weights saved successfully to '{model_save_path}'")

if __name__ == '__main__':
    train_gait_model()

Initializing custom training process...
Loading data from: D:\Study\Coding\Project\GAIT_2\New Dataset

Found 8 subjects: ['Adarsh', 'Aman', 'Ankit', 'Himanshu', 'Jyotibrat', 'Kunal', 'Swayam', 'Tiwari']
Saved class mapping to class_mapping.json
Training images: 4328
Validation images: 1849

Loading Xception model via TIMM...


d:\Study\Coding\Project\venv\Lib\site-packages\timm\models\_factory.py:138: UserWarning: Mapping deprecated model name xception to current legacy_xception.
  model = create_fn(
C:\Users\swaya\AppData\Local\Temp\ipykernel_28024\3646439441.py:82: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()


Training will use device: cuda:0

Epoch 1/20
----------


Train Phase:   0%|          | 0/68 [00:00<?, ?it/s]C:\Users\swaya\AppData\Local\Temp\ipykernel_28024\3646439441.py:108: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=use_amp):
Train Phase: 100%|██████████| 68/68 [00:20<00:00,  3.36it/s]


Train Loss: 1.6772 Acc: 0.3551



...

Validation Loss: 0.2480 Acc: 0.9297

Epoch 20/20
----------


Train Phase: 100%|██████████| 68/68 [00:19<00:00,  3.57it/s]


Train Loss: 0.0650 Acc: 0.9815



Validation Phase: 100%|██████████| 29/29 [00:13<00:00,  2.17it/s]

Validation Loss: 0.3295 Acc: 0.9200

Training complete in 10m 56s
Model weights saved successfully to 'new_dataset_gait_xception_FULL.pth'
